# Residual-Based Feature Boosting PoC

VS Code/Jupyter에서 위에서 아래로 실행하는 노트북입니다.

핵심 흐름:

1. `Xb` base feature로 baseline 수율 회귀 모델 학습
2. `baseline_residual = y - baseline_pred` 계산
3. defect별 bad/good group 기준으로 candidate feature 품질 필터링
4. candidate feature 하나만 사용해 current residual 예측
5. validation bad group의 `bad_rmse_reduction` 기준으로 feature 선택
6. 선택 feature를 `Xb + selected Xnew`에 추가해 final CatBoost 재학습
7. baseline vs final metric, ranking, SHAP summary 저장

중요: residual boosting 단계에서는 `Xb`를 다시 학습하지 않습니다. 후보 feature `x_j` 하나만 residual model에 사용합니다.

## 0. 환경 준비

**이 셀에서 하는 일**

- 현재 노트북이 어떤 폴더에서 실행되든 repo root를 찾아 작업 경로로 이동합니다.
- `feature_boosting` 패키지를 import할 수 있도록 `sys.path`를 설정합니다.
- 이후 셀에서 사용할 함수들을 미리 import합니다.

**입력**: 없음  
**출력**: `ROOT` 경로 출력



In [ ]:
from pathlib import Path
import os
import sys
import time

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "feature_boosting").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("repo root not found")
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from feature_boosting.answer_features import add_answer_feature_flags, answer_feature_mask
from feature_boosting.baseline_model import add_baseline_predictions, metrics_by_split, train_baseline_model
from feature_boosting.data_loader import align_base_and_candidates, candidate_feature_cols, load_base_dataset, load_base_feature_cols, load_candidate_features, load_group_ids, standardize_six_file_inputs
from feature_boosting.final_model import evaluate_model_by_groups, predict_final, train_final_model
from feature_boosting.reporting import baseline_residual_summary, final_metric_summary, plot_candidate_loss_ranking, plot_final_feature_set_summary, plot_final_metric_comparison, plot_residual_curve, plot_round_residual_points, prepare_output_dir, round_residual_summary, write_csv
from feature_boosting.residual_boosting import ResidualFeatureBooster, ResidualFeatureBoosterConfig
from feature_boosting.overfit import recommend_overfit_safe_settings
from feature_boosting.shap_analysis import compute_shap_summary
from feature_boosting.splitter import split_frame
from feature_boosting.validation import profile_candidate_features, validate_defect_groups, validate_input_columns
from feature_boosting.config import FeatureFilterConfig

print("ROOT =", ROOT)


## 1. 실험 설정

**이 셀에서 하는 일**

- toyset을 쓸지 실제 데이터를 쓸지 선택합니다.
- toyset 크기(`DEMO_N_WAFERS`, `DEMO_N_CANDIDATE_FEATURES`)를 정합니다.
- 실제 데이터 경로, defect bad/good group 경로, output 경로를 정의합니다.
- baseline/residual/final 모델 파라미터와 feature filter 기준을 정합니다.
- candidate scoring 진행률을 표시할지와 출력 간격을 정합니다.

기본값은 `USE_DEMO_DATA = True`입니다. 그래서 repo를 clone한 직후 실제 데이터가 없어도 toyset을 자동 생성해서 바로 실행됩니다.

실제 데이터가 준비되어 있으면 `USE_DEMO_DATA = False`로 바꾸고 아래 경로만 수정하면 됩니다.

실제 데이터가 6개 raw 파일 구조(`lot/wf/y`, candidate, base feature, defect별 `good_bad`)라면 `USE_RAW_SIX_FILE_DATA = True`로 바꾸고 `RAW_*` 경로와 컬럼명을 맞추면 됩니다.

CatBoost가 설치되어 있으면 `backend: "auto"`에서 CatBoost를 사용합니다. CatBoost만 강제하려면 `backend: "catboost"`로 바꾸세요.

**입력**: 사용자가 수정하는 설정값  
**출력**: `OUT_DIR` 생성 및 경로 출력



In [ ]:
USE_DEMO_DATA = True
USE_RAW_SIX_FILE_DATA = False  # Set True when using the real six-file raw input format.
DEMO_N_WAFERS = 2_000
DEMO_N_CANDIDATE_FEATURES = 100  # total candidate feature count, including hidden_defect_1/2
DEMO_RANDOM_SEED = 42
RUN_ID = "feature_boosting_rev0_notebook"

ID_COL = "sample_id"
TARGET_COL = "yield"
SPLIT_COL = "split"

# Default paths for already-standardized inputs.
BASE_DATASET_PATH = ROOT / "data" / "base_dataset.parquet"
CANDIDATE_FEATURES_PATH = ROOT / "data" / "candidate_features.parquet"
BASE_FEATURE_COLS_PATH = ROOT / "data" / "base_feature_cols.txt"
OUTPUT_BASE_DIR = ROOT / "outputs"

DEFECTS = [
    {
        "defect_id": "defect_1",
        "bad_group_path": ROOT / "data" / "groups" / "defect_1_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_1_good.csv",
    },
    {
        "defect_id": "defect_2",
        "bad_group_path": ROOT / "data" / "groups" / "defect_2_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_2_good.csv",
    },
    {
        "defect_id": "defect_3",
        "bad_group_path": ROOT / "data" / "groups" / "defect_3_bad.csv",
        "good_group_path": ROOT / "data" / "groups" / "defect_3_good.csv",
    },
]

# Six-file raw input mode. Change these paths and column names to match your files.
RAW_Y_PATH = ROOT / "data" / "raw" / "y.csv"
RAW_CANDIDATE_PATH = ROOT / "data" / "raw" / "candidate_features.csv"
RAW_BASE_FEATURE_PATH = ROOT / "data" / "raw" / "base_features.csv"
RAW_STANDARDIZED_DIR = ROOT / "data" / "standardized_from_raw"

RAW_Y_LOT_COL = "lot"
RAW_Y_WF_COL = "wf"
RAW_Y_TARGET_COL = "y"
RAW_CANDIDATE_LOT_COL = "lot"
RAW_CANDIDATE_WF_COL = "wf"
RAW_BASE_LOT_COL = "lot"
RAW_BASE_WF_COL = "wf"

# If the y file already has train/valid/test, put that column name here. Otherwise set None.
RAW_SPLIT_SOURCE_COL = None
RAW_TRAIN_RATIO = 0.6
RAW_VALID_RATIO = 0.2
RAW_SPLIT_SEED = 42

RAW_DEFECT_GROUPS = [
    {
        "defect_id": "defect_1",
        "group_path": ROOT / "data" / "raw" / "defect_1_good_bad.csv",
        "lot_col": "lot",
        "wf_col": "wf",
        "label_col": "good_bad",
        "good_value": "good",
        "bad_value": "bad",
    },
    {
        "defect_id": "defect_2",
        "group_path": ROOT / "data" / "raw" / "defect_2_good_bad.csv",
        "lot_col": "lot",
        "wf_col": "wf",
        "label_col": "good_bad",
        "good_value": "good",
        "bad_value": "bad",
    },
    {
        "defect_id": "defect_3",
        "group_path": ROOT / "data" / "raw" / "defect_3_good_bad.csv",
        "lot_col": "lot",
        "wf_col": "wf",
        "label_col": "good_bad",
        "good_value": "good",
        "bad_value": "bad",
    },
]

BASELINE_MODEL_PARAMS = {
    "backend": "auto",
    "iterations": 3000,
    "depth": 6,
    "learning_rate": 0.03,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
    "early_stopping_rounds": 100,
    "verbose": 200,
}

RESIDUAL_MODEL_PARAMS = {
    "backend": "auto",
    "iterations": 300,
    "depth": 3,
    "learning_rate": 0.05,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
    "early_stopping_rounds": 30,
    "verbose": False,
    "thread_count": 1,
}

FINAL_MODEL_PARAMS = dict(BASELINE_MODEL_PARAMS)

FEATURE_FILTER = FeatureFilterConfig(
    max_missing_rate=0.8,
    min_unique_values=2,
)

BOOSTING_N_ROUNDS = 5

# Feature selection knobs.
# Rank mode example: BOOSTING_SELECTION_MODE="top_k", SELECT_PER_ROUND=20
# Ratio threshold example: BOOSTING_SELECTION_MODE="threshold", BOOSTING_SELECTION_METRIC="bad_rmse_after_over_baseline", BOOSTING_SELECTION_THRESHOLD=0.3
SELECT_PER_ROUND = 1
BOOSTING_SELECTION_MODE = "top_k"  # "top_k" or "threshold"
BOOSTING_SELECTION_METRIC = "bad_rmse_reduction"
BOOSTING_SELECTION_THRESHOLD = None
BOOSTING_SELECTION_DIRECTION = "auto"
BOOSTING_MAX_SELECT_PER_ROUND = None
MIN_IMPROVEMENT = 0.0
MIN_VALID_BAD_SAMPLES = 1
BOOSTING_SHOW_PROGRESS = True
BOOSTING_PROGRESS_EVERY = 100

# Overfit guard: reject candidates that improve train residual but hurt validation.
AUTO_OVERFIT_SAFE_SETTINGS = True
OVERFIT_GUARD_ENABLED = True
OVERFIT_GUARD_METRIC_SCOPE = "bad"  # "bad", "good", or "global"
OVERFIT_GUARD_MIN_VALID_RMSE_REDUCTION = 0.0
OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE = 1.0
OVERFIT_GUARD_MAX_VALID_TRAIN_GAP = 0.25
OVERFIT_GUARD_USE_TEST = False  # exploratory only; True uses test as an additional guard
OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE = 1.05

# Candidate rank chart y-axis: after residual / base-feature-only baseline residual.
CANDIDATE_LOSS_GLOBAL_METRIC_COL = "valid_global_rmse_after_over_baseline"
CANDIDATE_LOSS_BAD_METRIC_COL = "valid_bad_rmse_after_over_baseline"

# Optional known answer features per defect.
# You can mix exact names and flexible matching rules.
# Examples:
# - "exact_feature_name"
# - {"contains_all": ["abc", "step2"]}  # feature name contains both abc and step2
# - {"contains_any": ["abc", "defect"]}
# - {"regex": r"abc.*step2"}
ANSWER_FEATURES = {
    "defect_1": ["hidden_defect_1", {"contains_all": ["hidden", "1"]}],
    "defect_2": ["hidden_defect_2"],
    "defect_3": [],
}

SHAP_ENABLED = True
SHAP_MAX_SAMPLES = 5000

OUT_DIR = prepare_output_dir(OUTPUT_BASE_DIR, RUN_ID)
print("OUT_DIR =", OUT_DIR)


## 2. 선택 사항: 데모 데이터 생성

**이 셀에서 하는 일**

- `USE_DEMO_DATA = True`이면 synthetic toyset을 생성합니다.
- `USE_RAW_SIX_FILE_DATA = True`이면 실제 6개 raw 파일을 표준 입력 파일로 변환합니다.
- wafer/sample row, baseline feature, candidate feature, target `yield`를 만듭니다.
- `hidden_defect_1`, `hidden_defect_2`는 residual에서 잡혀야 하는 planted signal입니다.
- defect별 bad/good sample list CSV를 생성하고, 이후 셀이 그 파일을 읽도록 경로를 바꿉니다.

`USE_DEMO_DATA = True`이면 아래 셀이 toyset CSV를 자동으로 생성합니다. clone 직후에는 이 기본값 그대로 실행하면 됩니다.

**입력**: `DEMO_N_WAFERS`, `DEMO_N_CANDIDATE_FEATURES`, `DEMO_RANDOM_SEED` 또는 `RAW_*` 경로/컬럼명  
**출력**: `data/residual_poc_demo/` 또는 `data/standardized_from_raw/` 아래 CSV 파일들



In [ ]:
if USE_DEMO_DATA and USE_RAW_SIX_FILE_DATA:
    raise ValueError("USE_DEMO_DATA and USE_RAW_SIX_FILE_DATA cannot both be True")

if USE_DEMO_DATA:
    demo_dir = ROOT / "data" / "residual_poc_demo"
    group_dir = demo_dir / "groups"
    group_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(DEMO_RANDOM_SEED)
    n = DEMO_N_WAFERS
    n_noise_features = max(0, DEMO_N_CANDIDATE_FEATURES - 2)
    sample_id = np.array([f"WF_{idx:05d}" for idx in range(n)])
    n_train = int(round(n * 0.6))
    n_valid = int(round(n * 0.2))
    n_test = n - n_train - n_valid
    split = np.array(["train"] * n_train + ["valid"] * n_valid + ["test"] * n_test)
    rng.shuffle(split)

    base_temp = rng.normal(0, 1, n)
    base_pressure = rng.normal(0, 1, n)
    hidden_defect_1 = rng.normal(0, 1, n)
    hidden_defect_2 = rng.normal(0, 1, n)
    noise_candidates = {f"cand_noise_{i:04d}": rng.normal(0, 1, n) for i in range(n_noise_features)}

    y = 80 + 5 * base_temp - 3 * base_pressure + 6 * hidden_defect_1 - 4 * hidden_defect_2 + rng.normal(0, 0.3, n)

    base_df = pd.DataFrame(
        {
            ID_COL: sample_id,
            TARGET_COL: y,
            SPLIT_COL: split,
            "base_temp": base_temp,
            "base_pressure": base_pressure,
        }
    )
    candidate_df = pd.DataFrame(
        {
            ID_COL: sample_id,
            "hidden_defect_1": hidden_defect_1,
            "hidden_defect_2": hidden_defect_2,
            **noise_candidates,
        }
    )

    BASE_DATASET_PATH = demo_dir / "base_dataset.csv"
    CANDIDATE_FEATURES_PATH = demo_dir / "candidate_features.csv"
    BASE_FEATURE_COLS_PATH = demo_dir / "base_feature_cols.txt"
    base_df.to_csv(BASE_DATASET_PATH, index=False, encoding="utf-8-sig")
    candidate_df.to_csv(CANDIDATE_FEATURES_PATH, index=False, encoding="utf-8-sig")
    BASE_FEATURE_COLS_PATH.write_text("base_temp\nbase_pressure\n", encoding="utf-8")

    bad1 = base_df.loc[hidden_defect_1 >= np.quantile(hidden_defect_1, 0.75), [ID_COL]]
    good1 = base_df.loc[hidden_defect_1 < np.quantile(hidden_defect_1, 0.50), [ID_COL]]
    bad2 = base_df.loc[hidden_defect_2 <= np.quantile(hidden_defect_2, 0.25), [ID_COL]]
    good2 = base_df.loc[hidden_defect_2 > np.quantile(hidden_defect_2, 0.50), [ID_COL]]
    bad3 = base_df.sample(min(max(1, n // 5), len(base_df)), random_state=DEMO_RANDOM_SEED)[[ID_COL]]
    good3_pool = base_df.drop(bad3.index)
    good3 = good3_pool.sample(min(max(1, n // 3), len(good3_pool)), random_state=DEMO_RANDOM_SEED + 1)[[ID_COL]]

    bad1.to_csv(group_dir / "defect_1_bad.csv", index=False, encoding="utf-8-sig")
    good1.to_csv(group_dir / "defect_1_good.csv", index=False, encoding="utf-8-sig")
    bad2.to_csv(group_dir / "defect_2_bad.csv", index=False, encoding="utf-8-sig")
    good2.to_csv(group_dir / "defect_2_good.csv", index=False, encoding="utf-8-sig")
    bad3.to_csv(group_dir / "defect_3_bad.csv", index=False, encoding="utf-8-sig")
    good3.to_csv(group_dir / "defect_3_good.csv", index=False, encoding="utf-8-sig")

    DEFECTS = [
        {"defect_id": "defect_1", "bad_group_path": group_dir / "defect_1_bad.csv", "good_group_path": group_dir / "defect_1_good.csv"},
        {"defect_id": "defect_2", "bad_group_path": group_dir / "defect_2_bad.csv", "good_group_path": group_dir / "defect_2_good.csv"},
        {"defect_id": "defect_3", "bad_group_path": group_dir / "defect_3_bad.csv", "good_group_path": group_dir / "defect_3_good.csv"},
    ]
    print("demo data written to", demo_dir)
    print("demo wafers:", len(base_df), "| demo candidate features:", candidate_df.shape[1] - 1)

elif USE_RAW_SIX_FILE_DATA:
    standardized = standardize_six_file_inputs(
        y_path=RAW_Y_PATH,
        candidate_path=RAW_CANDIDATE_PATH,
        base_feature_path=RAW_BASE_FEATURE_PATH,
        defect_groups=RAW_DEFECT_GROUPS,
        output_dir=RAW_STANDARDIZED_DIR,
        id_col=ID_COL,
        target_col=TARGET_COL,
        split_col=SPLIT_COL,
        y_lot_col=RAW_Y_LOT_COL,
        y_wf_col=RAW_Y_WF_COL,
        y_target_col=RAW_Y_TARGET_COL,
        candidate_lot_col=RAW_CANDIDATE_LOT_COL,
        candidate_wf_col=RAW_CANDIDATE_WF_COL,
        base_lot_col=RAW_BASE_LOT_COL,
        base_wf_col=RAW_BASE_WF_COL,
        split_source_col=RAW_SPLIT_SOURCE_COL,
        train_ratio=RAW_TRAIN_RATIO,
        valid_ratio=RAW_VALID_RATIO,
        split_seed=RAW_SPLIT_SEED,
    )
    BASE_DATASET_PATH = standardized["base_dataset"]
    CANDIDATE_FEATURES_PATH = standardized["candidate_features"]
    BASE_FEATURE_COLS_PATH = standardized["base_feature_cols"]
    DEFECTS = standardized["defects"]
    print("raw 6-file data standardized to", RAW_STANDARDIZED_DIR)
    print("base dataset:", BASE_DATASET_PATH)
    print("candidate features:", CANDIDATE_FEATURES_PATH)
    print("defects:", [item["defect_id"] for item in DEFECTS])

else:
    print("using existing standardized input paths")
    print("base dataset:", BASE_DATASET_PATH)
    print("candidate features:", CANDIDATE_FEATURES_PATH)
    print("base feature cols:", BASE_FEATURE_COLS_PATH)


## 3. 데이터 로드 및 검증

**이 셀에서 하는 일**

- base dataset, candidate feature dataset, base feature column list를 읽습니다.
- 필수 컬럼(`sample_id`, `yield`, `split`)이 있는지 확인합니다.
- base dataset과 candidate feature를 `sample_id` 기준으로 join합니다.
- `split` 컬럼을 기준으로 train/valid/test DataFrame을 나눕니다.

**입력**: base/candidate/base_feature_cols 파일  
**출력**: `all_df`, `train_df`, `valid_df`, `test_df`, `base_feature_cols`, `candidate_cols`



In [ ]:
base_df = load_base_dataset(BASE_DATASET_PATH)
candidate_df = load_candidate_features(CANDIDATE_FEATURES_PATH)
base_feature_cols = load_base_feature_cols(BASE_FEATURE_COLS_PATH)
candidate_cols = candidate_feature_cols(candidate_df, ID_COL)

validate_input_columns(
    base_df,
    candidate_df,
    base_feature_cols,
    id_col=ID_COL,
    target_col=TARGET_COL,
    split_col=SPLIT_COL,
)

all_df = align_base_and_candidates(base_df, candidate_df, ID_COL)
train_df, valid_df, test_df = split_frame(all_df, SPLIT_COL)

print("base_df:", base_df.shape)
print("candidate_df:", candidate_df.shape)
print("merged:", all_df.shape)
print("base features:", len(base_feature_cols))
print("candidate features:", len(candidate_cols))
print(all_df[SPLIT_COL].value_counts())

if AUTO_OVERFIT_SAFE_SETTINGS:
    overfit_recommendation = recommend_overfit_safe_settings(
        RESIDUAL_MODEL_PARAMS,
        n_train_rows=len(train_df),
        n_candidate_features=len(candidate_cols),
        n_base_features=len(base_feature_cols),
    )
    RESIDUAL_MODEL_PARAMS = overfit_recommendation["residual_model_params"]
    guard = overfit_recommendation["guard"]
    OVERFIT_GUARD_ENABLED = guard["overfit_guard_enabled"]
    OVERFIT_GUARD_METRIC_SCOPE = guard["overfit_guard_metric_scope"]
    OVERFIT_GUARD_MIN_VALID_RMSE_REDUCTION = guard["overfit_guard_min_valid_rmse_reduction"]
    OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE = guard["overfit_guard_max_valid_after_over_baseline"]
    OVERFIT_GUARD_MAX_VALID_TRAIN_GAP = guard["overfit_guard_max_valid_train_gap"]
    OVERFIT_GUARD_USE_TEST = guard["overfit_guard_use_test"]
    OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE = guard["overfit_guard_max_test_after_over_baseline"]
    overfit_preflight = pd.DataFrame(overfit_recommendation["summary_rows"])
    write_csv(overfit_preflight, OUT_DIR / "overfit_preflight_settings.csv")
    display(overfit_preflight)
    print("RESIDUAL_MODEL_PARAMS =", RESIDUAL_MODEL_PARAMS)

## 4. defect별 bad/good group 로드

**이 셀에서 하는 일**

- defect마다 bad wafer list와 good wafer list를 읽습니다.
- bad/good sample이 겹치지 않는지 검사합니다.
- valid/test split 안에 bad/good sample이 충분히 있는지 warning을 기록합니다.
- 이후 residual boosting에서 group metric을 계산할 수 있도록 `defect_groups` dict를 만듭니다.

**입력**: `DEFECTS`에 적힌 bad/good CSV 파일  
**출력**: `defect_groups`



In [ ]:
defect_groups = {}
for defect in DEFECTS:
    defect_id = defect["defect_id"]
    bad_ids = load_group_ids(defect["bad_group_path"], ID_COL)
    good_ids = load_group_ids(defect["good_group_path"], ID_COL)
    warnings = validate_defect_groups(
        defect_id,
        bad_ids,
        good_ids,
        all_df,
        id_col=ID_COL,
        split_col=SPLIT_COL,
        min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
    )
    defect_groups[defect_id] = {"bad": bad_ids, "good": good_ids, "warnings": set(warnings)}
    print(defect_id, "bad=", len(bad_ids), "good=", len(good_ids), "warnings=", warnings)

## 5. Baseline CatBoost 학습 및 residual 계산

**이 셀에서 하는 일**

- base feature `Xb`만 사용해 baseline 수율 회귀 모델을 학습합니다.
- 전체 row에 대해 `baseline_pred`를 생성합니다.
- `baseline_residual = yield - baseline_pred`를 계산합니다.
- split별 baseline metric과 defect group별 residual summary를 저장합니다.

**입력**: `train_df`, `valid_df`, `all_df`, `base_feature_cols`  
**출력**: `baseline_model`, `baseline_metrics`, `baseline_summary`, `baseline_pred`, `baseline_residual`



In [ ]:
baseline_model = train_baseline_model(
    train_df,
    valid_df,
    base_feature_cols,
    TARGET_COL,
    BASELINE_MODEL_PARAMS,
)

all_df = add_baseline_predictions(
    baseline_model,
    all_df,
    feature_cols=base_feature_cols,
    target_col=TARGET_COL,
)
train_df, valid_df, test_df = split_frame(all_df, SPLIT_COL)

baseline_metrics = metrics_by_split(all_df, target_col=TARGET_COL, pred_col="baseline_pred", split_col=SPLIT_COL)
baseline_summary = baseline_residual_summary(
    all_df,
    target_col=TARGET_COL,
    pred_col="baseline_pred",
    residual_col="baseline_residual",
    split_col=SPLIT_COL,
    id_col=ID_COL,
    defects=defect_groups,
)

write_csv(baseline_metrics, OUT_DIR / "baseline_metrics.csv")
write_csv(baseline_summary, OUT_DIR / "baseline_residual_summary.csv")

display(baseline_metrics)
display(baseline_summary.head(12))

## 6. defect별 residual feature boosting

**이 셀에서 하는 일**

- defect별로 candidate feature 품질을 먼저 검사합니다.
- 각 round마다 아직 선택되지 않은 candidate feature를 하나씩 residual model에 넣어 평가합니다.
- 선택되는 feature 수는 설정에 따라 1개, 상위 K개, 또는 threshold 통과 feature 전체가 될 수 있습니다.
- residual model은 `Xb`를 쓰지 않고 candidate feature `x_j` 하나만 사용합니다.
- 기본 feature 선택은 validation bad group의 `valid_bad_rmse_reduction` 기준이고, 설정에서 다른 metric으로 바꿀 수 있습니다.
- 노트북 설정에서 `top_k` 또는 `threshold` 방식으로 round별 선택 feature 수를 조절할 수 있습니다.
- candidate rank chart의 y축은 기본적으로 `after residual / baseline residual` 비율입니다.
- `ANSWER_FEATURES`에 defect별 정답인자를 넣으면 chart에 별도 마커로 표시됩니다.
- test metric은 선택에 쓰지 않고 기록/검증용으로만 저장합니다.
- round별 ranking, 선택 feature, residual curve를 CSV로 저장합니다.
- `BOOSTING_SHOW_PROGRESS = True`이면 candidate scoring 진행률이 출력됩니다.
- 각 round의 candidate별 after-boosting RMSE를 rank chart로 보여주고 PNG로 저장합니다.
- 모든 boosting이 끝난 뒤 defect별 round 평균 절대 residual point chart를 보여줍니다.

각 candidate feature는 `x_j -> current residual` 단일 feature 모델로만 평가됩니다. 선택 기준은 validation bad group의 residual 감소입니다.

**입력**: baseline prediction이 포함된 train/valid/test, candidate features, defect groups  
**출력**: `quality_summary`, `selected_features`, `residual_curve`, `round_mean_residual`, `rankings/*.csv`, `plots/*candidate_loss.png`, `plots/round_mean_abs_residual_points.png`



In [ ]:
standard_experiment_start_time = time.perf_counter()

booster = ResidualFeatureBooster(
    ResidualFeatureBoosterConfig(
        residual_model_params=RESIDUAL_MODEL_PARAMS,
        n_rounds=BOOSTING_N_ROUNDS,
        select_per_round=SELECT_PER_ROUND,
        main_metric="bad_rmse_reduction",
        min_improvement=MIN_IMPROVEMENT,
        selection_mode=BOOSTING_SELECTION_MODE,
        selection_metric=BOOSTING_SELECTION_METRIC,
        selection_threshold=BOOSTING_SELECTION_THRESHOLD,
        selection_direction=BOOSTING_SELECTION_DIRECTION,
        max_select_per_round=BOOSTING_MAX_SELECT_PER_ROUND,
        use_test_for_selection=False,
        min_valid_bad_samples=MIN_VALID_BAD_SAMPLES,
        overfit_guard_enabled=OVERFIT_GUARD_ENABLED,
        overfit_guard_metric_scope=OVERFIT_GUARD_METRIC_SCOPE,
        overfit_guard_min_valid_rmse_reduction=OVERFIT_GUARD_MIN_VALID_RMSE_REDUCTION,
        overfit_guard_max_valid_after_over_baseline=OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE,
        overfit_guard_max_valid_train_gap=OVERFIT_GUARD_MAX_VALID_TRAIN_GAP,
        overfit_guard_use_test=OVERFIT_GUARD_USE_TEST,
        overfit_guard_max_test_after_over_baseline=OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE,
        show_progress=BOOSTING_SHOW_PROGRESS,
        progress_every=BOOSTING_PROGRESS_EVERY,
    )
)

quality_frames = []
selected_frames = []
curve_frames = []

for defect in DEFECTS:
    defect_id = defect["defect_id"]
    groups = defect_groups[defect_id]
    quality = profile_candidate_features(
        all_df,
        candidate_cols,
        id_col=ID_COL,
        split_col=SPLIT_COL,
        bad_ids=groups["bad"],
        good_ids=groups["good"],
        config=FEATURE_FILTER,
        protected_cols={ID_COL, TARGET_COL, SPLIT_COL, *base_feature_cols},
    )
    quality.insert(0, "defect_id", defect_id)
    quality_frames.append(quality)

    if any(str(item).startswith("low_valid_bad_samples") for item in groups.get("warnings", set())):
        print(defect_id, "skipped: low valid bad samples")
        continue

    answer_rules = ANSWER_FEATURES.get(defect_id, [])
    answer_tracking_cols = []
    if answer_rules:
        candidate_series = pd.Series(candidate_cols, dtype=str)
        answer_tracking_cols = candidate_series[answer_feature_mask(candidate_series, answer_rules)].tolist()
    print(f"[{defect_id}] answer tracking cols: {answer_tracking_cols}")
    if answer_rules and not answer_tracking_cols:
        print(f"[WARN {defect_id}] ANSWER_FEATURES did not match any candidate column. Check names/rules: {answer_rules}")

    print(f"[{defect_id}] scoring {len(candidate_cols)} candidates")
    result = booster.run_for_defect(
        train_df=train_df,
        valid_df=valid_df,
        test_df=test_df,
        candidate_cols=candidate_cols,
        target_col=TARGET_COL,
        id_col=ID_COL,
        baseline_pred_col="baseline_pred",
        defect_id=defect_id,
        bad_sample_ids=groups["bad"],
        good_sample_ids=groups["good"],
        quality_summary=quality,
        output_dir=OUT_DIR / "rankings",
    )

    if result.rankings:
        for ranking_df in result.rankings:
            if ranking_df.empty or "round" not in ranking_df.columns:
                continue
            ranking_df = add_answer_feature_flags(ranking_df, feature_col="feature_name", rules=answer_rules)
            round_no = int(ranking_df["round"].max())
            answer_mask = ranking_df["is_answer_feature"].fillna(False).astype(bool) if "is_answer_feature" in ranking_df.columns else pd.Series(False, index=ranking_df.index)
            answer_rows = ranking_df[answer_mask].copy()
            if answer_rules and answer_rows.empty:
                print(f"[WARN {defect_id} round {round_no}] no answer feature row in ranking. answer_tracking_cols={answer_tracking_cols}")
            elif not answer_rows.empty:
                answer_cols_to_show = [
                    col for col in [
                        "defect_id", "round", "rank", "feature_name", "selected", "eligible_for_selection",
                        "ranking_only", "already_selected", "valid_bad_rmse_after_over_baseline",
                        "valid_global_rmse_after_over_baseline", "fail_reason", "overfit_guard_reason",
                        "answer_match_rule",
                    ] if col in answer_rows.columns
                ]
                display(answer_rows[answer_cols_to_show])
            write_csv(ranking_df, OUT_DIR / "rankings" / f"{defect_id}_round_{round_no}.csv")
            fig = plot_candidate_loss_ranking(
                ranking_df,
                output_path=OUT_DIR / "plots" / f"{defect_id}_round_{round_no}_candidate_loss.png",
                global_metric_col=CANDIDATE_LOSS_GLOBAL_METRIC_COL,
                bad_metric_col=CANDIDATE_LOSS_BAD_METRIC_COL,
                answer_features=answer_rules,
                title_prefix=f"{defect_id} round {round_no}",
            )
            if fig is not None:
                display(fig)

    if not result.selected_features.empty:
        selected_for_defect = add_answer_feature_flags(result.selected_features, feature_col="feature_name", rules=answer_rules)
        selected_frames.append(selected_for_defect)
        display(selected_for_defect)
    else:
        print(defect_id, "selected no feature")

    if not result.residual_curve.empty:
        curve_for_defect = add_answer_feature_flags(result.residual_curve, feature_col="selected_feature", rules=answer_rules)
        curve_frames.append(curve_for_defect)

quality_summary = pd.concat(quality_frames, ignore_index=True) if quality_frames else pd.DataFrame()
selected_features = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame()
residual_curve = pd.concat(curve_frames, ignore_index=True) if curve_frames else pd.DataFrame()

write_csv(quality_summary, OUT_DIR / "candidate_quality_summary.csv")
write_csv(selected_features, OUT_DIR / "selected_features.csv")
write_csv(residual_curve, OUT_DIR / "residual_reduction_curve.csv")
plot_residual_curve(residual_curve, OUT_DIR, answer_features_by_defect=ANSWER_FEATURES)

round_mean_residual = round_residual_summary(residual_curve, baseline_summary, group="bad")
write_csv(round_mean_residual, OUT_DIR / "round_mean_residual_summary.csv")
fig = plot_round_residual_points(
    round_mean_residual,
    output_path=OUT_DIR / "plots" / "round_mean_abs_residual_points.png",
    answer_features_by_defect=ANSWER_FEATURES,
)
if fig is not None:
    display(fig)

display(selected_features)
display(residual_curve)

standard_elapsed_seconds = time.perf_counter() - standard_experiment_start_time
standard_runtime_summary = pd.DataFrame(
    [
        {
            "method": "standard_feature_by_feature",
            "elapsed_seconds": standard_elapsed_seconds,
            "elapsed_minutes": standard_elapsed_seconds / 60.0,
            "n_candidate_features": len(candidate_cols),
            "n_selected_features": len(selected_features),
            "n_defects": len(DEFECTS),
        }
    ]
)
write_csv(standard_runtime_summary, OUT_DIR / "standard_runtime_summary.csv")
display(standard_runtime_summary)


## 7. Final model: Xb + selected Xnew 재학습

**이 셀에서 하는 일**

- residual boosting에서 선택된 feature 목록을 중복 없이 정리합니다.
- 최종 feature set을 `base_feature_cols + selected_cols`로 만듭니다.
- final CatBoost 회귀 모델을 새로 학습합니다.
- baseline 모델과 final 모델의 train/valid/test 및 defect bad/good group 성능을 비교합니다.
- final feature 구성을 막대 차트로 보여줍니다.
- baseline vs final RMSE/MAE를 split별 비교 차트로 보여줍니다.

**입력**: `selected_features`, `base_feature_cols`, `all_df`  
**출력**: `final_model`, `final_pred`, `final_model_metrics.csv`, `final_model_metric_summary.csv`, `plots/final_model_metric_comparison.png`



In [ ]:
selected_cols = []
if not selected_features.empty:
    for feature in selected_features["feature_name"].dropna().astype(str):
        if feature in candidate_cols and feature not in selected_cols:
            selected_cols.append(feature)

final_feature_cols = base_feature_cols + selected_cols
final_feature_summary = pd.DataFrame(
    [
        {"feature_type": "base", "count": len(base_feature_cols)},
        {"feature_type": "selected", "count": len(selected_cols)},
    ]
)
write_csv(final_feature_summary, OUT_DIR / "final_feature_set_summary.csv")

print("selected_cols:", selected_cols)
print("final feature count:", len(final_feature_cols))
display(final_feature_summary)
fig = plot_final_feature_set_summary(
    final_feature_summary,
    output_path=OUT_DIR / "plots" / "final_feature_set_summary.png",
)
if fig is not None:
    display(fig)

final_model = train_final_model(
    train_df,
    valid_df,
    feature_cols=final_feature_cols,
    target_col=TARGET_COL,
    catboost_params=FINAL_MODEL_PARAMS,
)
all_df["final_pred"] = predict_final(final_model, all_df, final_feature_cols)

final_metrics = pd.concat(
    [
        evaluate_model_by_groups(
            all_df,
            model_name="baseline",
            pred_col="baseline_pred",
            target_col=TARGET_COL,
            split_col=SPLIT_COL,
            id_col=ID_COL,
            defects=defect_groups,
        ),
        evaluate_model_by_groups(
            all_df,
            model_name="final",
            pred_col="final_pred",
            target_col=TARGET_COL,
            split_col=SPLIT_COL,
            id_col=ID_COL,
            defects=defect_groups,
        ),
    ],
    ignore_index=True,
)
write_csv(final_metrics, OUT_DIR / "final_model_metrics.csv")

final_metric_summary_df = final_metric_summary(final_metrics)
write_csv(final_metric_summary_df, OUT_DIR / "final_model_metric_summary.csv")
fig = plot_final_metric_comparison(
    final_metric_summary_df,
    output_path=OUT_DIR / "plots" / "final_model_metric_comparison.png",
)
if fig is not None:
    display(fig)

display(final_metric_summary_df)
display(final_metrics)


## 8. Optional SHAP 검증

**이 셀에서 하는 일**

- final model 기준으로 SHAP summary를 계산합니다.
- 전체 test sample, defect별 bad group, defect별 good group을 나누어 feature importance를 기록합니다.
- residual boosting으로 선택된 feature가 final model에서도 상위권에 나타나는지 확인할 수 있게 합니다.
- CatBoost/SHAP 계산이 불가능한 환경에서는 skip 상태를 CSV에 남깁니다.

SHAP은 원인 확정이 아니라, 선택 feature가 final model에서 실제로 사용되는지 확인하는 보조 검증입니다.

**입력**: `final_model`, test set, `selected_features`  
**출력**: `shap_summary.csv`



In [ ]:
if SHAP_ENABLED:
    shap_summary = compute_shap_summary(
        final_model,
        all_df[all_df[SPLIT_COL].astype(str) == "test"],
        feature_cols=final_feature_cols,
        id_col=ID_COL,
        defects=defect_groups,
        selected_features=selected_features,
        max_samples=SHAP_MAX_SAMPLES,
    )
else:
    shap_summary = pd.DataFrame([{"status": "disabled"}])

write_csv(shap_summary, OUT_DIR / "shap_summary.csv")
display(shap_summary.head(30))

## 9. 산출물 확인

**이 셀에서 하는 일**

- 이번 run의 output directory를 출력합니다.
- 저장된 CSV, model, plot 파일 목록을 보여줍니다.
- 노트북 실행 후 어떤 결과물이 생겼는지 빠르게 확인하는 마지막 점검 셀입니다.

**입력**: `OUT_DIR`  
**출력**: 저장된 산출물 목록



In [ ]:
print("saved to:", OUT_DIR)
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUT_DIR))

## 10. Second experiment: block preselection residual boosting

**? ??? ?? ?**

- ?? feature-by-feature boosting ??? ??? ???.
- candidate feature? `BLOCK_SIZE`?? ?? block residual model? ?? ?????.
- stage 1?? ?? ?? block 1?? ????.
- stage 2?? ??? block ?? feature? 1?? ?? ??/?????.
- ??? feature ??? ???? `top_k` ?? `threshold` ???? ?????.

**??**: `OUT_DIR / "block_experiment"` ?? block ranking, block ?? feature ranking, ?? ?? feature CSV? plot


In [ ]:
from feature_boosting.metrics import mae, rmse, reduction
from feature_boosting.modeling import fit_regressor, predict_regressor

BLOCK_EXPERIMENT_ENABLED = True
BLOCK_SIZE = 10
BLOCK_STAGE1_SELECT_BLOCKS = 1
BLOCK_STAGE1_SELECTION_METRIC = BOOSTING_SELECTION_METRIC
BLOCK_FINAL_SELECTION_MODE = BOOSTING_SELECTION_MODE  # "top_k" or "threshold"
BLOCK_FINAL_SELECTION_METRIC = BOOSTING_SELECTION_METRIC
BLOCK_FINAL_TOP_K = SELECT_PER_ROUND
BLOCK_FINAL_THRESHOLD = BOOSTING_SELECTION_THRESHOLD
BLOCK_EXPERIMENT_DIR = OUT_DIR / "block_experiment"
BLOCK_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print("BLOCK_EXPERIMENT_DIR =", BLOCK_EXPERIMENT_DIR)
print("BLOCK_SIZE =", BLOCK_SIZE)
print("BLOCK_FINAL_SELECTION_MODE =", BLOCK_FINAL_SELECTION_MODE)


In [ ]:
def _block_metric_name(metric, *, use_test=False):
    if metric in {"bad_rmse_reduction", "rmse_reduction"}:
        return "test_bad_rmse_reduction" if use_test else "valid_bad_rmse_reduction"
    if metric.startswith("valid_") or metric.startswith("test_") or metric.startswith("train_"):
        return metric
    return f"valid_{metric}"


def _block_higher_is_better(metric):
    lower_metric = str(metric).lower()
    if "over_baseline" in lower_metric or lower_metric.endswith("_after") or lower_metric.endswith("_ratio"):
        return False
    return True


def _safe_ratio(numerator, denominator):
    if not np.isfinite(numerator) or not np.isfinite(denominator) or denominator == 0:
        return float("nan")
    return float(numerator / denominator)


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


def _mask_ids(df, sample_ids):
    return df[ID_COL].astype(str).isin(sample_ids).to_numpy()


def _add_block_split_metrics(row, *, prefix, df, y, before_pred, after_pred, baseline_pred, groups):
    masks = {
        "bad": _mask_ids(df, groups["bad"]),
        "good": _mask_ids(df, groups["good"]),
        "global": np.ones(len(df), dtype=bool),
    }
    for group_name, mask in masks.items():
        before_rmse = rmse(y[mask], before_pred[mask])
        after_rmse = rmse(y[mask], after_pred[mask])
        baseline_rmse = rmse(y[mask], baseline_pred[mask])
        before_mae = mae(y[mask], before_pred[mask])
        after_mae = mae(y[mask], after_pred[mask])
        baseline_mae = mae(y[mask], baseline_pred[mask])
        row[f"{prefix}_{group_name}_rmse_before"] = before_rmse
        row[f"{prefix}_{group_name}_rmse_after"] = after_rmse
        row[f"{prefix}_{group_name}_rmse_reduction"] = reduction(before_rmse, after_rmse)
        row[f"{prefix}_{group_name}_rmse_baseline"] = baseline_rmse
        row[f"{prefix}_{group_name}_rmse_after_over_baseline"] = _safe_ratio(after_rmse, baseline_rmse)
        row[f"{prefix}_{group_name}_mae_before"] = before_mae
        row[f"{prefix}_{group_name}_mae_after"] = after_mae
        row[f"{prefix}_{group_name}_mae_reduction"] = reduction(before_mae, after_mae)
        row[f"{prefix}_{group_name}_mae_baseline"] = baseline_mae
        row[f"{prefix}_{group_name}_mae_after_over_baseline"] = _safe_ratio(after_mae, baseline_mae)


def _apply_block_overfit_guard(row):
    scope = OVERFIT_GUARD_METRIC_SCOPE
    row["overfit_guard_scope"] = scope
    if row.get("fail_reason"):
        row["overfit_guard_pass"] = False
        row["overfit_guard_reason"] = f"not_evaluated:{row.get('fail_reason')}"
        row["overfit_gap_valid_train_rmse_ratio"] = np.nan
        return row
    if not OVERFIT_GUARD_ENABLED:
        row["overfit_guard_pass"] = True
        row["overfit_guard_reason"] = ""
        return row

    reasons = []
    valid_reduction_col = f"valid_{scope}_rmse_reduction"
    valid_ratio_col = f"valid_{scope}_rmse_after_over_baseline"
    train_ratio_col = f"train_{scope}_rmse_after_over_baseline"
    test_ratio_col = f"test_{scope}_rmse_after_over_baseline"

    valid_reduction = _safe_float(row.get(valid_reduction_col))
    if OVERFIT_GUARD_MIN_VALID_RMSE_REDUCTION is not None and (
        not np.isfinite(valid_reduction) or valid_reduction <= float(OVERFIT_GUARD_MIN_VALID_RMSE_REDUCTION)
    ):
        reasons.append(f"{valid_reduction_col}<={float(OVERFIT_GUARD_MIN_VALID_RMSE_REDUCTION):g}")

    valid_ratio = _safe_float(row.get(valid_ratio_col))
    if OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE is not None and (
        not np.isfinite(valid_ratio) or valid_ratio > float(OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE)
    ):
        reasons.append(f"{valid_ratio_col}>{float(OVERFIT_GUARD_MAX_VALID_AFTER_OVER_BASELINE):g}")

    train_ratio = _safe_float(row.get(train_ratio_col))
    gap = valid_ratio - train_ratio if np.isfinite(valid_ratio) and np.isfinite(train_ratio) else float("nan")
    row["overfit_gap_valid_train_rmse_ratio"] = gap
    if OVERFIT_GUARD_MAX_VALID_TRAIN_GAP is not None and np.isfinite(gap) and gap > float(OVERFIT_GUARD_MAX_VALID_TRAIN_GAP):
        reasons.append(f"valid_train_{scope}_rmse_ratio_gap>{float(OVERFIT_GUARD_MAX_VALID_TRAIN_GAP):g}")

    if OVERFIT_GUARD_USE_TEST:
        test_ratio = _safe_float(row.get(test_ratio_col))
        if OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE is not None and np.isfinite(test_ratio) and test_ratio > float(OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE):
            reasons.append(f"{test_ratio_col}>{float(OVERFIT_GUARD_MAX_TEST_AFTER_OVER_BASELINE):g}")

    row["overfit_guard_pass"] = not reasons
    row["overfit_guard_reason"] = ";".join(reasons)
    return row


def _score_residual_feature_set(feature_cols, *, defect_id, groups, stage, feature_set_name):
    row = {
        "defect_id": defect_id,
        "stage": stage,
        "feature_set_name": feature_set_name,
        "feature_name": feature_set_name,
        "n_features": len(feature_cols),
        "feature_names": "|".join(feature_cols),
        "fail_reason": "",
    }
    missing = [col for col in feature_cols if col not in train_df.columns]
    if missing:
        row["fail_reason"] = f"missing_column:{missing[:3]}"
        return _apply_block_overfit_guard(row)

    y_train = train_df[TARGET_COL].to_numpy(dtype=float)
    y_valid = valid_df[TARGET_COL].to_numpy(dtype=float)
    y_test = test_df[TARGET_COL].to_numpy(dtype=float)
    base_train = train_df["baseline_pred"].to_numpy(dtype=float)
    base_valid = valid_df["baseline_pred"].to_numpy(dtype=float)
    base_test = test_df["baseline_pred"].to_numpy(dtype=float)
    residual_train = y_train - base_train
    residual_valid = y_valid - base_valid

    try:
        model = fit_regressor(train_df, residual_train, valid_df, residual_valid, feature_cols, RESIDUAL_MODEL_PARAMS)
        pred_train_delta = predict_regressor(model, train_df, feature_cols)
        pred_valid_delta = predict_regressor(model, valid_df, feature_cols)
        pred_test_delta = predict_regressor(model, test_df, feature_cols)
    except Exception as exc:
        row["fail_reason"] = f"model_fit_failed:{type(exc).__name__}"
        return _apply_block_overfit_guard(row)

    _add_block_split_metrics(
        row,
        prefix="train",
        df=train_df,
        y=y_train,
        before_pred=base_train,
        after_pred=base_train + pred_train_delta,
        baseline_pred=base_train,
        groups=groups,
    )
    _add_block_split_metrics(
        row,
        prefix="valid",
        df=valid_df,
        y=y_valid,
        before_pred=base_valid,
        after_pred=base_valid + pred_valid_delta,
        baseline_pred=base_valid,
        groups=groups,
    )
    _add_block_split_metrics(
        row,
        prefix="test",
        df=test_df,
        y=y_test,
        before_pred=base_test,
        after_pred=base_test + pred_test_delta,
        baseline_pred=base_test,
        groups=groups,
    )
    return _apply_block_overfit_guard(row)


def _rank_block_rows(frame, *, metric_col, higher_is_better):
    if frame.empty:
        return frame
    result = frame.copy()
    result[metric_col] = pd.to_numeric(result[metric_col], errors="coerce") if metric_col in result.columns else np.nan
    result = result.sort_values([metric_col, "feature_set_name"], ascending=[not higher_is_better, True], na_position="last").reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result.insert(1, "ranking_metric", metric_col)
    return result


def _select_from_ranked_frame(frame, *, metric_col, mode, top_k, threshold, higher_is_better):
    if frame.empty or metric_col not in frame.columns:
        return frame.iloc[0:0].copy()
    eligible = frame[
        frame["fail_reason"].fillna("").astype(str).eq("")
        & frame["overfit_guard_pass"].fillna(False).astype(bool)
        & pd.to_numeric(frame[metric_col], errors="coerce").notna()
    ].copy()
    if eligible.empty:
        return eligible
    mode = str(mode).strip().lower()
    if mode in {"threshold", "metric_threshold", "ratio_threshold"}:
        if threshold is None:
            print(f"[BLOCK EXPERIMENT] threshold mode selected, but threshold is None. No feature selected.")
            return eligible.iloc[0:0].copy()
        values = pd.to_numeric(eligible[metric_col], errors="coerce")
        keep = values >= float(threshold) if higher_is_better else values <= float(threshold)
        return eligible[keep].copy()
    selected = eligible.head(max(1, int(top_k))).copy()
    if higher_is_better:
        selected = selected[pd.to_numeric(selected[metric_col], errors="coerce") > MIN_IMPROVEMENT]
    elif threshold is not None:
        selected = selected[pd.to_numeric(selected[metric_col], errors="coerce") <= float(threshold)]
    return selected


In [ ]:
if BLOCK_EXPERIMENT_ENABLED:
    block_experiment_start_time = time.perf_counter()
    block_quality_frames = []
    block_ranking_frames = []
    block_feature_ranking_frames = []
    block_selected_feature_frames = []

    block_metric_col = _block_metric_name(BLOCK_STAGE1_SELECTION_METRIC, use_test=False)
    block_higher_is_better = _block_higher_is_better(block_metric_col)
    feature_metric_col = _block_metric_name(BLOCK_FINAL_SELECTION_METRIC, use_test=False)
    feature_higher_is_better = _block_higher_is_better(feature_metric_col)

    for defect in DEFECTS:
        defect_id = defect["defect_id"]
        groups = defect_groups[defect_id]
        answer_rules = ANSWER_FEATURES.get(defect_id, [])

        quality = profile_candidate_features(
            all_df,
            candidate_cols,
            id_col=ID_COL,
            split_col=SPLIT_COL,
            bad_ids=groups["bad"],
            good_ids=groups["good"],
            config=FEATURE_FILTER,
            protected_cols={ID_COL, TARGET_COL, SPLIT_COL, *base_feature_cols},
        )
        quality.insert(0, "defect_id", defect_id)
        block_quality_frames.append(quality)
        passing_candidates = quality.loc[quality["is_pass"].astype(bool), "feature_name"].astype(str).tolist()
        blocks = [passing_candidates[idx : idx + BLOCK_SIZE] for idx in range(0, len(passing_candidates), BLOCK_SIZE)]
        print(f"[{defect_id} block experiment] passing candidates={len(passing_candidates)}, blocks={len(blocks)}")
        if not blocks:
            continue

        block_rows = []
        for block_idx, block_cols in enumerate(blocks, start=1):
            block_name = f"block_{block_idx:04d}"
            row = _score_residual_feature_set(block_cols, defect_id=defect_id, groups=groups, stage="block", feature_set_name=block_name)
            row["block_id"] = block_name
            row["block_start_feature"] = block_cols[0] if block_cols else ""
            row["block_end_feature"] = block_cols[-1] if block_cols else ""
            row["contains_answer_feature"] = bool(answer_rules and answer_feature_mask(pd.Series(block_cols, dtype=str), answer_rules).any())
            block_rows.append(row)
            if block_idx == 1 or block_idx == len(blocks) or block_idx % 10 == 0:
                print(f"[{defect_id} block experiment] scored block {block_idx}/{len(blocks)}")

        block_ranking = _rank_block_rows(pd.DataFrame(block_rows), metric_col=block_metric_col, higher_is_better=block_higher_is_better)
        block_ranking["selected_block"] = False
        selected_blocks = _select_from_ranked_frame(
            block_ranking,
            metric_col=block_metric_col,
            mode="top_k",
            top_k=BLOCK_STAGE1_SELECT_BLOCKS,
            threshold=None,
            higher_is_better=block_higher_is_better,
        )
        selected_block_ids = set(selected_blocks.get("block_id", pd.Series(dtype=str)).astype(str))
        block_ranking["selected_block"] = block_ranking["block_id"].astype(str).isin(selected_block_ids)
        write_csv(block_ranking, BLOCK_EXPERIMENT_DIR / f"{defect_id}_block_ranking.csv")
        block_ranking_frames.append(block_ranking)
        display(block_ranking.head(20))

        block_plot_frame = block_ranking.rename(columns={"block_id": "feature_name"}).copy()
        fig = plot_candidate_loss_ranking(
            block_plot_frame,
            output_path=BLOCK_EXPERIMENT_DIR / f"{defect_id}_block_ranking.png",
            global_metric_col="valid_global_rmse_after_over_baseline",
            bad_metric_col="valid_bad_rmse_after_over_baseline",
            selected_col="selected_block",
            title_prefix=f"{defect_id} block ranking",
        )
        if fig is not None:
            display(fig)

        if selected_blocks.empty:
            print(f"[{defect_id} block experiment] selected no block")
            continue

        selected_block = selected_blocks.iloc[0]
        selected_block_id = str(selected_block["block_id"])
        selected_block_features = str(selected_block["feature_names"]).split("|")
        print(f"[{defect_id} block experiment] selected block: {selected_block_id}, n_features={len(selected_block_features)}")

        feature_rows = []
        for feature in selected_block_features:
            row = _score_residual_feature_set([feature], defect_id=defect_id, groups=groups, stage="feature_in_selected_block", feature_set_name=feature)
            row["block_id"] = selected_block_id
            row["feature_name"] = feature
            feature_rows.append(row)
        feature_ranking = _rank_block_rows(pd.DataFrame(feature_rows), metric_col=feature_metric_col, higher_is_better=feature_higher_is_better)
        feature_ranking = add_answer_feature_flags(feature_ranking, feature_col="feature_name", rules=answer_rules)
        selected_feature_rows = _select_from_ranked_frame(
            feature_ranking,
            metric_col=feature_metric_col,
            mode=BLOCK_FINAL_SELECTION_MODE,
            top_k=BLOCK_FINAL_TOP_K,
            threshold=BLOCK_FINAL_THRESHOLD,
            higher_is_better=feature_higher_is_better,
        )
        selected_feature_names = set(selected_feature_rows.get("feature_name", pd.Series(dtype=str)).astype(str))
        feature_ranking["selected"] = feature_ranking["feature_name"].astype(str).isin(selected_feature_names)
        write_csv(feature_ranking, BLOCK_EXPERIMENT_DIR / f"{defect_id}_selected_block_feature_ranking.csv")
        block_feature_ranking_frames.append(feature_ranking)
        display(feature_ranking)

        fig = plot_candidate_loss_ranking(
            feature_ranking,
            output_path=BLOCK_EXPERIMENT_DIR / f"{defect_id}_selected_block_feature_ranking.png",
            global_metric_col="valid_global_rmse_after_over_baseline",
            bad_metric_col="valid_bad_rmse_after_over_baseline",
            answer_features=answer_rules,
            title_prefix=f"{defect_id} features inside {selected_block_id}",
        )
        if fig is not None:
            display(fig)

        if not selected_feature_rows.empty:
            selected_feature_rows = selected_feature_rows.copy()
            selected_feature_rows["defect_id"] = defect_id
            selected_feature_rows["block_id"] = selected_block_id
            selected_feature_rows["selection_mode"] = BLOCK_FINAL_SELECTION_MODE
            block_selected_feature_frames.append(selected_feature_rows)
            display(selected_feature_rows)
        else:
            print(f"[{defect_id} block experiment] selected no feature inside {selected_block_id}")

    block_quality_summary = pd.concat(block_quality_frames, ignore_index=True) if block_quality_frames else pd.DataFrame()
    block_rankings = pd.concat(block_ranking_frames, ignore_index=True) if block_ranking_frames else pd.DataFrame()
    block_feature_rankings = pd.concat(block_feature_ranking_frames, ignore_index=True) if block_feature_ranking_frames else pd.DataFrame()
    block_selected_features = pd.concat(block_selected_feature_frames, ignore_index=True) if block_selected_feature_frames else pd.DataFrame()

    write_csv(block_quality_summary, BLOCK_EXPERIMENT_DIR / "block_candidate_quality_summary.csv")
    write_csv(block_rankings, BLOCK_EXPERIMENT_DIR / "block_rankings.csv")
    write_csv(block_feature_rankings, BLOCK_EXPERIMENT_DIR / "selected_block_feature_rankings.csv")
    write_csv(block_selected_features, BLOCK_EXPERIMENT_DIR / "block_experiment_selected_features.csv")

    display(block_selected_features)
    print("block experiment saved to:", BLOCK_EXPERIMENT_DIR)

    def _selected_feature_list(frame, *, defect_id, feature_col="feature_name"):
        if frame is None or frame.empty or feature_col not in frame.columns or "defect_id" not in frame.columns:
            return []
        values = frame.loc[frame["defect_id"].astype(str) == str(defect_id), feature_col].dropna().astype(str).tolist()
        return list(dict.fromkeys(values))

    feature_comparison_rows = []
    for defect in DEFECTS:
        defect_id = defect["defect_id"]
        standard_features = _selected_feature_list(selected_features, defect_id=defect_id) if "selected_features" in globals() else []
        block_features = _selected_feature_list(block_selected_features, defect_id=defect_id)
        overlap = sorted(set(standard_features) & set(block_features))
        feature_comparison_rows.append(
            {
                "defect_id": defect_id,
                "standard_selected_features": ", ".join(standard_features),
                "block_selected_features": ", ".join(block_features),
                "overlap_features": ", ".join(overlap),
                "n_standard_selected": len(standard_features),
                "n_block_selected": len(block_features),
                "n_overlap": len(overlap),
            }
        )
    experiment_selected_feature_comparison = pd.DataFrame(feature_comparison_rows)
    write_csv(experiment_selected_feature_comparison, BLOCK_EXPERIMENT_DIR / "experiment_selected_feature_comparison.csv")
    display(experiment_selected_feature_comparison)

    def _last_standard_row(defect_id):
        if "selected_features" not in globals() or selected_features.empty:
            return None
        work = selected_features[selected_features["defect_id"].astype(str) == str(defect_id)].copy()
        if work.empty:
            return None
        work["round"] = pd.to_numeric(work["round"], errors="coerce")
        return work.sort_values("round").iloc[-1]

    residual_comparison_rows = []
    block_final_feature_sets_scored = 0
    for defect in DEFECTS:
        defect_id = defect["defect_id"]
        standard_row = _last_standard_row(defect_id)
        standard_features = _selected_feature_list(selected_features, defect_id=defect_id) if "selected_features" in globals() else []
        if standard_row is not None:
            residual_comparison_rows.append(
                {
                    "defect_id": defect_id,
                    "method": "standard_feature_by_feature",
                    "metric_source": "last cumulative selected round",
                    "selected_features": ", ".join(standard_features),
                    "valid_bad_rmse_after": standard_row.get("valid_bad_rmse_after", np.nan),
                    "valid_bad_rmse_after_over_baseline": standard_row.get("valid_bad_rmse_after_over_baseline", np.nan),
                    "test_bad_rmse_after": standard_row.get("test_bad_rmse_after", np.nan),
                    "test_bad_rmse_after_over_baseline": standard_row.get("test_bad_rmse_after_over_baseline", np.nan),
                    "valid_global_rmse_after": standard_row.get("valid_global_rmse_after", np.nan),
                    "valid_global_rmse_after_over_baseline": standard_row.get("valid_global_rmse_after_over_baseline", np.nan),
                }
            )
        else:
            residual_comparison_rows.append(
                {
                    "defect_id": defect_id,
                    "method": "standard_feature_by_feature",
                    "metric_source": "no selected feature",
                    "selected_features": "",
                }
            )

        block_features = _selected_feature_list(block_selected_features, defect_id=defect_id)
        if block_features:
            block_set_row = _score_residual_feature_set(
                block_features,
                defect_id=defect_id,
                groups=defect_groups[defect_id],
                stage="block_selected_feature_set",
                feature_set_name="block_selected_features",
            )
            block_final_feature_sets_scored += 1
            residual_comparison_rows.append(
                {
                    "defect_id": defect_id,
                    "method": "block_preselection_then_feature",
                    "metric_source": "selected feature set refit",
                    "selected_features": ", ".join(block_features),
                    "valid_bad_rmse_after": block_set_row.get("valid_bad_rmse_after", np.nan),
                    "valid_bad_rmse_after_over_baseline": block_set_row.get("valid_bad_rmse_after_over_baseline", np.nan),
                    "test_bad_rmse_after": block_set_row.get("test_bad_rmse_after", np.nan),
                    "test_bad_rmse_after_over_baseline": block_set_row.get("test_bad_rmse_after_over_baseline", np.nan),
                    "valid_global_rmse_after": block_set_row.get("valid_global_rmse_after", np.nan),
                    "valid_global_rmse_after_over_baseline": block_set_row.get("valid_global_rmse_after_over_baseline", np.nan),
                }
            )
        else:
            residual_comparison_rows.append(
                {
                    "defect_id": defect_id,
                    "method": "block_preselection_then_feature",
                    "metric_source": "no selected feature",
                    "selected_features": "",
                }
            )
    experiment_residual_comparison = pd.DataFrame(residual_comparison_rows)
    write_csv(experiment_residual_comparison, BLOCK_EXPERIMENT_DIR / "experiment_residual_comparison.csv")
    display(experiment_residual_comparison)

    block_elapsed_seconds = time.perf_counter() - block_experiment_start_time
    standard_elapsed_value = globals().get("standard_elapsed_seconds", np.nan)
    experiment_runtime_comparison = pd.DataFrame(
        [
            {
                "method": "standard_feature_by_feature",
                "elapsed_seconds": standard_elapsed_value,
                "elapsed_minutes": standard_elapsed_value / 60.0 if np.isfinite(standard_elapsed_value) else np.nan,
                "n_candidate_features": len(candidate_cols),
                "n_selected_features": len(selected_features) if "selected_features" in globals() else np.nan,
                "stage1_units_scored": np.nan,
                "stage2_units_scored": np.nan,
                "final_feature_sets_scored": np.nan,
            },
            {
                "method": "block_preselection_then_feature",
                "elapsed_seconds": block_elapsed_seconds,
                "elapsed_minutes": block_elapsed_seconds / 60.0,
                "n_candidate_features": len(candidate_cols),
                "n_selected_features": len(block_selected_features),
                "stage1_units_scored": len(block_rankings),
                "stage2_units_scored": len(block_feature_rankings),
                "final_feature_sets_scored": block_final_feature_sets_scored,
            },
        ]
    )
    write_csv(experiment_runtime_comparison, BLOCK_EXPERIMENT_DIR / "experiment_runtime_comparison.csv")
    display(experiment_runtime_comparison)
